# Допуск: Sequence to Sequence Learning with Neural Networks

**Статья:** I. Sutskever, O. Vinyals, Q. V. Le, 2014 — <https://arxiv.org/abs/1409.3215>

5 заданий. Под заголовком каждого задания указан его тип:

- **что есть что** — названия функций или классов заменены: определите, что чему
  соответствует, и ответьте на вопросы;
- **расхождение со статьёй** — код реализует не то, что описано в статье: найдите это место;
- **перепутаны местами блоки кода** — какой-то блок стоит не на своём месте: найдите его;
- **неверный комментарий** — код корректен, но один комментарий утверждает неправду: найдите
  его.

Для найденного отклонения напишите: где оно, что не так, какое место статьи (раздел /
формула / таблица / цитата) нарушено и как должно быть. Код писать не нужно — достаточно
объяснить словами.

Общие импорты:

In [ ]:
import random
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence

PAD, UNK, BOS, EOS = 0, 1, 2, 3
SPECIALS = ["<pad>", "<unk>", "<bos>", "<eos>"]

Сниппеты — части одной реализации; функции, которые используются, но не показаны, считайте корректными.

## Задание 1. Три способа подать пару предложений

**Тип задания:** что есть что — названия функций/классов заменены, определите, что чему соответствует.

Функции превращают пару «предложение — перевод» в последовательности для энкодера и
декодера. Определите, какая функция соответствует **основной модели статьи**, какая — модели
**без разворота** (с ней авторы сравниваются), а какая — варианту, которого в статье **нет**.
Ошибок в коде нет.

In [ ]:
def pair_a(src, tgt):
    return src[::-1] + ["<eos>"], tgt + ["<eos>"]

def pair_b(src, tgt):
    return src + ["<eos>"], tgt[::-1] + ["<eos>"]

def pair_c(src, tgt):
    return src + ["<eos>"], tgt + ["<eos>"]

def time_lags(pair_fn, n):
    # toy "translation" with monotonic alignment: source word k <-> target word k
    s, t = pair_fn(list(range(n)), list(range(n)))
    pos_s = {w: i for i, w in enumerate(s) if w != "<eos>"}
    pos_t = {w: len(s) + i for i, w in enumerate(t) if w != "<eos>"}  # target follows source
    lags = [pos_t[k] - pos_s[k] for k in range(n)]
    return sum(lags) / n, min(lags)

**Вопросы.**
1. Запустите `time_lags` для трёх функций при n = 5 и n = 20. Подтверждается ли §3.3: при
   развороте «the average distance between corresponding words … is unchanged», но «minimal
   time lag is greatly reduced»?
2. По этой метрике вариант, которого нет в статье, не отличается от основного. Почему тогда
   разворачивают именно source?

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 2. Модель

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Найдите одно отклонение от статьи.

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, emb=1000, hidden=1000, layers=4):
        super().__init__()
        self.src_emb = nn.Embedding(src_vocab, emb)
        self.tgt_emb = nn.Embedding(tgt_vocab, emb)
        self.encoder = nn.LSTM(emb, hidden, layers, batch_first=True)
        self.decoder = self.encoder
        self.proj = nn.Linear(hidden, tgt_vocab)     # softmax over the target vocabulary

    def encode(self, src):                           # src: (B, S)
        _, state = self.encoder(self.src_emb(src))
        return state                                 # (h, c), each (layers, B, H)

    def decode(self, tgt_in, state):                 # tgt_in: (B, T)
        out, state = self.decoder(self.tgt_emb(tgt_in), state)
        return self.proj(out), state                 # (B, T, V)

    def forward(self, src, tgt_in):
        return self.decode(tgt_in, self.encode(src))[0]

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 3. Шаг обучения

**Тип задания:** перепутаны местами блоки кода.

`seq2seq_loss` — функция потерь (корректна, не показана). Найдите одно отклонение.

In [ ]:
def train_step(model, opt, src, tgt):
    # opt = torch.optim.SGD(model.parameters(), lr=0.7)
    opt.zero_grad()
    loss = seq2seq_loss(model, src, tgt)
    loss.backward()
    opt.step()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)   # if ||g|| > 5: g = 5 g / ||g||
    return loss.item()

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 4. Beam search

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Найдите одно отклонение от процедуры декодирования, описанной в §3.2.

In [ ]:
@torch.no_grad()
def beam_search(model, src, B=12, max_len=100):
    # src: (1, S)
    beam = [([BOS], 0.0, model.encode(src))]       # (prefix, log p(prefix | S), decoder state)
    complete = []
    for _ in range(max_len):
        candidates = []
        for prefix, score, state in beam:
            logits, new_state = model.decode(src.new_tensor([[prefix[-1]]]), state)
            logp = logits[0, -1].log_softmax(-1)
            top = logp.topk(B)
            for lp, w in zip(top.values.tolist(), top.indices.tolist()):
                candidates.append((prefix + [w], score + lp, new_state))
        candidates.sort(key=lambda h: h[1], reverse=True)
        beam = candidates[:B]
        complete += [h for h in beam if h[0][-1] == EOS]
    return max(complete or beam, key=lambda h: h[1])[0]

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 5. Rescoring n-best списка

**Тип задания:** неверный комментарий — код корректен, но один комментарий утверждает неправду.

`token_logprobs` возвращает (N, T) — log p(y_t | v, y_<t) для каждой позиции каждой гипотезы
(функция корректна). Найдите одно отклонение от статьи.

In [ ]:
@torch.no_grad()
def rescore(model, src, hyps, smt_scores):
    # hyps: (N, T) n-best list of the SMT baseline, smt_scores: (N,)
    # LSTM score = average log-probability per token, so long hypotheses are not penalized
    logp = token_logprobs(model, src.expand(len(hyps), -1), hyps)     # (N, T)
    lstm = (logp * (hyps != PAD)).sum(1)                               # (N,)
    return hyps[(0.5 * lstm + 0.5 * smt_scores).argmax()]              # even average

**Ваш ответ:**

_Напишите ответ здесь._